### Импорты

In [7]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import importlib

import judge as judge_mod
importlib.reload(judge_mod)
from judge import JudgeModel

PROC = Path('../data/processed')
RESULTS = Path('../results')

train_df = pd.read_parquet(PROC / 'train.parquet')
val_df   = pd.read_parquet(PROC / 'val.parquet')
test_df  = pd.read_parquet(PROC / 'test.parquet')

print(f"train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

train: 3,787,046  val: 371,021  test: 474,130


### Загрузка items_en.csv и обучение судьи

In [8]:
# 1. Поиск items_en.csv в известных местах
possible_paths = [
    PROC / 'items_en.csv',
    PROC.parent / 'raw' / 'items_en.csv',
    PROC.parent.parent / 'data' / 'raw' / 'items_en.csv',
]
items_path = next((p for p in possible_paths if p.exists()), None)
if items_path is None:
    raise FileNotFoundError(f"items_en.csv not found; tried {possible_paths}")

items_df = pd.read_csv(items_path)
print(f"[judge] items_en.csv: {items_df.shape}, columns={items_df.columns.tolist()}")

# 2. Обучение судьи на train, валидация на val (ранняя остановка)
print("[judge] обучение на train ...")
judge = JudgeModel()
judge.fit(train_df, items_df, val_df=val_df)
judge.save(RESULTS / 'checkpoints_judge' / 'judge')
print(f"[judge] сохранено: {RESULTS / 'checkpoints_judge' / 'judge.txt'}")

# 3. Качество судьи на тесте — 50k сэмплов, чтобы быстро
print("[judge] качество на test ...")
judge_metrics = judge.evaluate_on_df(test_df, sample_n=50000)
print(f"  MAE={judge_metrics['mae']:.4f}  RMSE={judge_metrics['rmse']:.4f}  R²={judge_metrics['r2']:.4f}")
print(f"  Для сравнения RM v2 test: MAE=0.3308  RMSE=0.3816  R²=0.0609")

[judge] items_en.csv: (15963, 17), columns=['Unnamed: 0', 'item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'description', 'keywords', 'actors_translated', 'actors_transliterated', 'directors_translated', 'transliterated']
[judge] обучение на train ...


c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[judge] сохранено: ..\results\checkpoints_judge\judge.txt
[judge] качество на test ...
  MAE=0.3383  RMSE=0.4103  R²=-0.0784
  Для сравнения RM v2 test: MAE=0.3308  RMSE=0.3816  R²=0.0609
